# Filter Design

This notebook extracts the receive-chain bandpass design material from `dsp.ipynb`. It focuses on cutoff choice, order, and how the transfer function reshapes both signal and audio.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Order Changes the Transition Band

Higher-order filters roll off faster, but they also become more selective and can add more ringing or delay. The legacy notebook compared Butterworth orders directly; this notebook makes that comparison reusable.

In [ ]:
orders = [2, 4, 8]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

for order in orders:
    sos = signal.butter(order, [300, 3000], btype="band", fs=WORK_FS, output="sos")
    w, h = signal.sosfreqz(sos, worN=8192, fs=WORK_FS)
    axes[0].plot(w, 20 * np.log10(np.abs(h) + 1e-12), label=f"order {order}")
    axes[1].plot(w, np.unwrap(np.angle(h)), label=f"order {order}")

axes[0].set_xlim(0, 8000)
axes[0].set_ylim(-80, 5)
axes[0].set_title("Magnitude Response")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Magnitude (dB)")
axes[0].legend()
axes[1].set_xlim(0, 8000)
axes[1].set_title("Phase Response")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Radians")
axes[1].legend()
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_design(order=4, low_cut=300.0, high_cut=3000.0):
    axes[0].clear()
    axes[1].clear()
    sos = signal.butter(order, [low_cut, high_cut], btype="band", fs=WORK_FS, output="sos")
    w, h = signal.sosfreqz(sos, worN=8192, fs=WORK_FS)
    filtered = normalize(signal.sosfilt(sos, voice_work))
    axes[0].plot(w, 20 * np.log10(np.abs(h) + 1e-12), color="tab:green")
    axes[0].set_xlim(0, 8000)
    axes[0].set_ylim(-80, 5)
    axes[0].set_title("Designed Filter Response")
    plot_spectrum(filtered, fs=WORK_FS, ax=axes[1], title="Filtered Voice Spectrum", color="tab:red")
    axes[1].set_xlim(0, 8000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(filtered, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_design,
    order=int_slider(min_value=2, max_value=10, step=2, value=4, description="Order"),
    low_cut=float_slider(min_value=100, max_value=1000, step=50, value=300, description="Low Hz"),
    high_cut=float_slider(min_value=1500, max_value=5000, step=100, value=3000, description="High Hz"),
)
display(controls, audio_out)


## Key Takeaway

Filter design is a tradeoff problem, not a single "best" choice. Order and cutoff control selectivity, delay, and how natural the recovered audio sounds.